# Bit-identity validation

This notebook verifies that the refactored `codenames_interpretability` package produces outputs interchangeable with the existing N=2000 runs already saved in Drive. Run this before trusting the refactored code with a new experiment.

**What it does.** Picks a model, samples 50 boards under the contract seed, runs the refactored extraction pipeline to a temporary output directory, then compares the resulting `*_general_{mode}.csv` rows against the corresponding rows of the existing N=2000 outputs in `--against`. Reports `Validation PASSED` if every numeric column matches within the configured tolerance.

**Default model:** BERT-base-uncased (fastest, smallest hardware requirement). Edit `MODEL` below to validate against a different model's existing run.

**What to do on failure.** Read the printed mismatch list. The most common drift sources are documented in `CONTEXT.md` Section 6 (random-state consumption, hot-loop operation order, dictionary iteration order, fp16 conversion timing, tokenizer init args). Fix the package code, push, re-run this notebook.

## Setup (run cells 1–3 once per session)

In [ ]:
# Cell 1 — Clone or update the package code from GitHub.
import os
REPO_URL = "https://github.com/JoaoPedroFPK/codenames-interpretability.git"
REPO_DIR = "/content/codenames-interpretability"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
# Cell 2 — Install the package in editable mode.
!pip install -q -e {REPO_DIR}

In [ ]:
# Cell 3 — Autoreload and mount Drive.
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount("/content/drive")

## Configuration

Set `MODEL` and `AGAINST_DIR` to the model whose existing N=2000 run you want to validate against. `N` is the number of boards to validate (default 50).

In [ ]:
MODEL = "bert"  # one of: mistral, qwen, qwen_random, bert, bert_random, t5, modernbert
DATASET_PATH = "/content/drive/MyDrive/TCC/clue_generation.csv"
AGAINST_DIR = "/content/drive/MyDrive/TCC/bert_outputs"
N = 50
TOLERANCE = 1e-6

## Path A — One-shot CLI invocation

Run the CLI's `validate` subcommand. Output is identical to the cell-by-cell path below.

In [ ]:
!codenames-experiment validate \
    --model {MODEL} \
    --dataset {DATASET_PATH} \
    --against {AGAINST_DIR} \
    -n {N} \
    --tolerance {TOLERANCE}

## Path B — Step-by-step validation

Use this if you want to inspect intermediate state — the refactored DataFrames before comparison, the diff itself, etc.

In [ ]:
# Cell 4 — Imports.
import importlib, tempfile, os
import numpy as np
import pandas as pd

from codenames_interpretability.contract import CONTRACT_V1, Contract
from codenames_interpretability.data import load_dataset, sample_turns
from codenames_interpretability.generation import generate_response
from codenames_interpretability.loop import run_extraction
from codenames_interpretability.cli import MODEL_REGISTRY

In [ ]:
# Cell 5 — Resolve and load the chosen model.
mod_path, attr = MODEL_REGISTRY[MODEL]
loader = getattr(importlib.import_module(mod_path), attr)
model, tokenizer, meta = loader()
print(f"Validating model: {meta['model_name']} (prefix={meta['prefix']})")

In [ ]:
# Cell 6 — Sample N boards under the contract seed.
df = load_dataset(DATASET_PATH)
df_sample = sample_turns(df, n=N, seed=CONTRACT_V1.random_seed)
row_ids = sorted(df_sample["row_id"].tolist())
print(f"Validation sample: {len(row_ids)} boards")
print(f"First 10 row_ids: {row_ids[:10]}")

In [ ]:
# Cell 7 — Run the refactored pipeline to a temp directory.
contract = Contract(
    sample_size=N,
    candidate_order=CONTRACT_V1.candidate_order,
    pooling_methods=CONTRACT_V1.pooling_methods,
    vector_subsample_size=CONTRACT_V1.vector_subsample_size,
    n_shuffles=CONTRACT_V1.n_shuffles,
    generation_max_tokens=CONTRACT_V1.generation_max_tokens,
    shard_boards=CONTRACT_V1.shard_boards,
    random_seed=CONTRACT_V1.random_seed,
    max_seq_len=CONTRACT_V1.max_seq_len,
)

tmpdir = tempfile.mkdtemp(prefix="cnames_validate_")
results = run_extraction(
    model=model, tokenizer=tokenizer, df=df_sample,
    base_dir=tmpdir, prefix=meta["prefix"], contract=contract,
    chat_template_strategy=meta["chat_template_strategy"],
    forward_hidden_states_mode=meta["forward_hidden_states_mode"],
    use_truncation=meta["use_truncation"],
    num_layers=meta["num_layers"], hidden_dim=meta["hidden_dim"],
    device=meta["device"],
    has_generation=meta["supports_generation"],
    generation_fn=generate_response if meta["supports_generation"] else None,
)
print(f"Refactored outputs written to: {tmpdir}")

In [ ]:
# Cell 8 — Compare against the existing N=2000 outputs.
failures = []

for mode_name in ["no_social", "with_social"]:
    new_general = results[mode_name]["general_df"]
    old_path = os.path.join(AGAINST_DIR, f"{meta['prefix']}_general_{mode_name}.csv")
    if not os.path.exists(old_path):
        failures.append(f"Missing reference file: {old_path}")
        continue
    old_general = pd.read_csv(old_path)
    old_general = old_general[old_general["row_id"].isin(row_ids)]

    for col in new_general.select_dtypes(include=[np.number]).columns:
        if col not in old_general.columns:
            continue
        merged = new_general[["row_id", "permutation_id", col]].merge(
            old_general[["row_id", "permutation_id", col]],
            on=["row_id", "permutation_id"],
            suffixes=("_new", "_old"),
        )
        if merged.empty:
            continue
        diff = (merged[f"{col}_new"] - merged[f"{col}_old"]).abs()
        max_diff = float(diff.max(skipna=True))
        if max_diff > TOLERANCE:
            failures.append(
                f"general[{mode_name}][{col}]: max diff {max_diff:.2e} (tolerance {TOLERANCE:.0e})"
            )

if failures:
    print("\nValidation FAILED:")
    for line in failures:
        print(f"  - {line}")
else:
    print("\nValidation PASSED")